# Week 2 Day 2 Lab - Agentic RAG, MCP and Memory

## Giving the agent a long-term memory

Another 25 claims have arrived at UnderCover, and most of them turn on something you won't find in the claim: when a storm was named, what a device is, or how deep a dive site goes.

We give an agent a long-term memory: Qdrant, a vector database, running as an MCP server. We teach it the facts it's missing, then run the UnderCover agent with and without that memory as experiments in Langfuse.

## 4 parts:

PART 1: The even harder claims

PART 2: A memory over MCP

PART 3: Teaching the agent

PART 4: Experiments in Langfuse

**Hugo Volckaert's version.** This is the Week 2 Day 2 lab with the three challenges done at the end: a word lookup, `text-find`, in place of Qdrant; a memory about each customer; and a research sub-agent with `qdrant-find` and the customer notes. `decide` counts the results of `text-find` as well as `qdrant-find`, and the conclusions from my runs are at the end, before the clean-up.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls GPT-6 Luna through OpenRouter and costs a few cents. If you'd rather not spend anything, there's a free version of this lab in <a href="../../labs/week2day2free.ipynb">week2day2free.ipynb</a>.<br/><br/>
            You'll need your Langfuse keys from Day 1. Teaching the agent takes about 3 minutes, and each experiment takes 3 or 4 minutes to run. For a quicker run, set <code>TINY = True</code> in Part 4 to use 5 claims in place of 25.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path
from typing import Literal

import httpx
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler
sys.path.append("../../labs")
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter and Langfuse

The keys come from the `.env` file in the `undercover` folder, including the Langfuse keys you added on Day 1.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
langfuse = get_client()

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

## Part 1: The even harder claims

The new claims are in `data/evals/even_harder`, with their policy records and the handlers' decisions.

In [ ]:
EVEN_HARDER = Path("../../data/evals/even_harder")


def load(name):
    """Read one of the JSON files of the even harder claims."""
    return json.loads((EVEN_HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

### A lost bag in Tokyo

Here's a claim on a policy with Electronics Plus, which raises the limit for phones, tablets, laptops, cameras, e-readers and smartwatches. What is a Halden Arc?

In [ ]:
tokyo = [claim for claim in claims if claim["claim_id"] == "CLM-1307"][0]

print(tokyo["story"])
pd.DataFrame(tokyo["items"])

This is how the handler decided it.

In [ ]:
decision = labels["CLM-1307"]

print(decision["reason"])
pd.DataFrame(decision["calculation"])

### The knowledge base

The facts the handlers used are in `data/knowledge.json`: news about storms and strikes, a catalogue of devices, and notes on dive sites, rivers and activity operators. Some of them aren't needed by any claim.

In [ ]:
knowledge = json.loads(Path("../../data/knowledge.json").read_text(encoding="utf-8"))

pd.set_option("display.max_colwidth", None)
pd.DataFrame(knowledge)

## Part 2: A memory over MCP

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">MCP</h2>
            <span style="color:#118AB2;">MCP, the Model Context Protocol, is an open standard for connecting agents to tools. An MCP server offers tools, and an MCP client makes them available to an agent.<br/><br/>
            Qdrant's MCP server has two tools: <code>qdrant-store</code> saves a piece of text in a vector database, and <code>qdrant-find</code> searches it by meaning. We run it on our own machine with <code>uvx</code>, and the database lives in a local folder.
            </span>
        </td>
    </tr>
</table>

### One adjustment for Windows

When Python launches another program, it hands that program a place to write its error messages. Inside a Jupyter notebook on Windows, that place is not a real file, and the library that launches MCP servers trips over it. The cell below points the MCP server's error log somewhere safe before we connect. On Mac and Linux it does nothing at all, so run it either way and carry on.

In [ ]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

### Connecting to the memory server

Qdrant keeps its database in the folder `qdrant_data`, which we first make sure is empty. The first time you connect, `uvx` downloads the server and a small embedding model which can take half a minute.

The `--python 3.12.12` argument ensures that a compatible Python version is used (this is a workaround for a known Qdrant MCP server issue). Thank you to student Faisal A. for finding this.

In [ ]:
MEMORY = Path("qdrant_data").resolve()
shutil.rmtree(MEMORY, ignore_errors=True)

In [ ]:
MEMORY_ENV = {"QDRANT_LOCAL_PATH": str(MEMORY), "COLLECTION_NAME": "memory"}

parameters = {"transport": "stdio", "command": "uvx", "args": ["--python", "3.12.12", "mcp-server-qdrant@0.8.1"], "env": MEMORY_ENV}

client = MultiServerMCPClient({ "memory": parameters})

memory_tools = await client.get_tools()
for memory_tool in memory_tools:
    print(f"{memory_tool.name}: {memory_tool.description}\n")

Each tool call starts the server, runs the tool and stops the server again. Qdrant's local mode lets only one process open the database at a time, so when the agent searches twice at once, one of the searches fails. `ToolRetryMiddleware` tries a failed tool call again after a short wait.

### An agent with a memory

The memory tools come from the server; the instructions tell the agent how to use them.

The MCP tools only run asynchronously so we need to use async python. In Part 4, Langfuse runs each experiment on its own event loop, and a connection opened on one event loop can't be reused on another, so we give the model an HTTP client that doesn't keep connections open between calls.

In [ ]:
MEMORY_INSTRUCTIONS = """You have a long-term memory.
When you are told a fact, store it in your memory word for word.
When you are asked a question, search your memory and answer from what you find. If your memory has nothing on it, say that you don't know.
"""

http_client = httpx.AsyncClient(limits=httpx.Limits(max_keepalive_connections=0))
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5, http_async_client=http_client)

assistant = create_agent(model=luna, tools=memory_tools, system_prompt=MEMORY_INSTRUCTIONS, middleware=[ToolRetryMiddleware(max_retries=3)])

`ask` starts a new conversation every time, so the agent can't answer from earlier messages. Whatever it knows next time has to come from its memory.

In [ ]:
async def ask(message):
    """Send a message to the assistant in a new conversation and return its reply."""
    result = await assistant.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["messages"][-1].text


print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

## Part 3: Teaching the agent

Now we tell the agent each fact in the knowledge base, one message at a time. This takes about 3 minutes.

In [ ]:
for entry in knowledge:
    await ask(f"Please remember this: {entry['fact']}")
    print(f"Told: {entry['fact'][:80]}...")

We ask the same questions again, each in a new conversation.

In [ ]:
print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

A tool is still a function, so we can search the memory ourselves. The search works on meaning, so a question can find a fact that shares none of its words. It returns the 10 closest facts, nearest first.

In [ ]:
search = [memory_tool for memory_tool in memory_tools if memory_tool.name == "qdrant-find"][0]

results = await search.ainvoke({"query": "a gadget for watching films on a wall"})
for line in json.loads(results[0]["text"]):
    print(line)

## Part 4: Experiments in Langfuse

### The UnderCover agent

The instructions, the prompt, `record_decision` and the payout tool are the same as on Day 1.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

### Two agents

Both agents run Luna at `low` with the payout tool. The second one also gets `qdrant-find`, so it can search the memory whenever it decides to. This is agentic RAG: the agent chooses when to retrieve and what to search for. Its server runs in read-only mode, so it can search the memory but can't change it.

In [ ]:
read_only_client = MultiServerMCPClient({
    "memory": {"transport": "stdio", "command": "uvx", "args": ["--python", "3.12.12", "mcp-server-qdrant@0.8.1"], "env": {**MEMORY_ENV, "QDRANT_READ_ONLY": "true"}}
})

memory_search = await read_only_client.get_tools()
print([memory_tool.name for memory_tool in memory_search])

In [ ]:
MEMORY_LINE = "Before you decide, search your memory for any product, event or place in the claim that you need to know more about.\n"

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")

memory_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, *memory_search],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)

### Scoring

The scores are the ones from Day 1, with one more for retrieval. For the claims that need a fact from the knowledge base, `found_fact` is whether the agent's searches brought that fact back.

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def found_fact(answer, expected):
    """For a claim that needs a fact from the knowledge base, whether the agent's searches found it."""
    if not expected["needs"]:
        return None
    return all(fact in answer["retrieved"] for fact in expected["needs"])


def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()

In [ ]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, the cost of its mistake, and whether it found the fact it needed."""
    evaluations = [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]
    found = found_fact(output, expected_output)
    if found is not None:
        evaluations.append(Evaluation(name="found_fact", value=int(found)))
    return evaluations


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "found_fact": found_fact(answer, expected),
            "llm_cost": answer["cost"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "found_fact": table["found_fact"].dropna().astype(float).mean(),
        "llm_cost": table["llm_cost"].sum(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]


def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()

### The dataset

We upload the even harder claims as a Langfuse dataset, with the handler's decision and the facts each claim needs as the expected output.

In [ ]:
DATASET = "even-harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's even harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"], "needs": label["needs"]},
        metadata={"tiny": label["tiny"]},
    )

As on Day 1, `TINY = True` keeps 5 of the claims, and `N_TIMES` is the number of times each experiment runs.

In [ ]:
TINY = False
N_TIMES = 1

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims")

### The task

`decide` uses `ainvoke`, since the memory tools only run asynchronously, and Langfuse runs async tasks too. Along with the decision, `decide` keeps what the agent's memory searches returned.

In [ ]:
langfuse_handler = CallbackHandler()

# The tools whose results count as memory searches: Qdrant's, and the word lookup from the challenge
SEARCH_TOOLS = {"qdrant-find", "text-find"}


async def decide(agent, claim):
    """Run an agent on one claim and return its decision, the cost of its model calls and what its memory searches returned."""
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [langfuse_handler]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    retrieved = "\n".join(message.text for message in result["messages"] if message.type == "tool" and message.name in SEARCH_TOOLS)
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "cost": cost, "retrieved": retrieved}


async def payout_task(*, item, **kwargs):
    """Run the agent with the payout tool on one claim from the dataset."""
    return await decide(payout_agent, item.input)


async def memory_task(*, item, **kwargs):
    """Run the agent with the payout tool and memory on one claim from the dataset."""
    return await decide(memory_agent, item.input)

### Experiment 1: the payout tool

This is the agent from Day 1 with the payout tool. On Day 1 we turned Luna's reasoning off, and today both agents run with Luna at `low`. Each run takes about 3 minutes.

In [ ]:
payout_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool",
        description="The agent with the payout tool",
        data=items,
        task=payout_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    payout_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
payout_table = pd.concat(payout_tables, ignore_index=True)
show_confusion_matrix(payout_table, "Payout tool")

In [ ]:
len(payout_table[~payout_table["right"]])

### Experiment 2: the payout tool and memory

The same agent, now able to search its memory. Each run takes about 4 minutes.

In [ ]:
memory_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool and memory",
        description="The agent with the payout tool and a memory of the knowledge base",
        data=items,
        task=memory_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    memory_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
memory_table = pd.concat(memory_tables, ignore_index=True)
show_confusion_matrix(memory_table, "Payout tool and memory")

In [ ]:
memory_table[~memory_table["right"]]

Open a trace from this experiment in Langfuse to see what the agent searched for and what came back.

### Comparing the experiments

In [ ]:
pd.DataFrame([average_scores(payout_table), average_scores(memory_table)], index=["Payout tool", "Payout tool and memory"]).round(3)

In Langfuse, open **Datasets** and then `even-harder-claims`, and select the runs to compare them claim by claim.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Agentic RAG allows you to give an agent expertise about your business, and to build up its knowledge over time (long-term memory).
            Consider the expertise that you could equip an agent with for your business.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: replace Qdrant with a simple word lookup. Write a tool that searches the facts in <code>data/knowledge.json</code> for the words in its query, give it to the memory agent in place of <code>qdrant-find</code>, and run Experiment 2 again. <code>decide</code> picks out the search results by the tool's name, so change <code>qdrant-find</code> there to the name of your tool. How do the scores compare with searching by meaning?<br/><br/>
            Then give the agent a memory about each customer: a Markdown file for each one, named by the claim's <code>customer_id</code>, with a tool to read a customer's notes and a tool to add to them. Each customer has only one claim in this set, so write a few notes yourself to start with. With <code>N_TIMES</code> above 1, what should happen to the notes between runs?<br/><br/>
            Stretch challenge: move the retrieval into a sub-agent that has <code>qdrant-find</code> and the customer memory tools, and let the UnderCover agent call the sub-agent as a tool.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

## Challenge 1: `text-find`, a word lookup in place of Qdrant

`text-find` searches the facts in `data/knowledge.json` for the words in its query. It drops short common words like "the" and "of", counts how many of the query's words each fact contains, and returns the facts with the most matches first, up to 10, as `qdrant-find` does. It matches whole words only: no meaning, no plurals, no synonyms.

The facts come back word for word, and `decide` counts the results of both `qdrant-find` and `text-find`, so `found_fact` still works.

In [ ]:
import re

STOP_WORDS = {
    "a", "an", "the", "and", "or", "of", "in", "on", "at", "to", "for", "by", "with", "from", "as", "about",
    "is", "are", "was", "were", "be", "do", "does", "did", "it", "its", "this", "that", "any",
    "what", "when", "how", "which", "who", "my", "our", "your",
}
MAX_RESULTS = 10


def words(text):
    """The lowercase words in a text, without the stop words."""
    return {word for word in re.findall(r"[a-z0-9]+", text.lower()) if word not in STOP_WORDS}


@tool("text-find", parse_docstring=True)
def text_find(query: str) -> str:
    """Search the knowledge base for facts that contain the words in the query: products, storms, strikes, dive sites, rivers and activity operators.

    Args:
        query: The words to look for, such as a product's name or a place
    """
    query_words = words(query)
    matches = [(len(query_words & words(entry["fact"])), index, entry["fact"]) for index, entry in enumerate(knowledge)]
    matches = sorted((match for match in matches if match[0] > 0), key=lambda match: (-match[0], match[1]))
    if not matches:
        return f"No facts found for the query '{query}'"
    return "\n".join([f"Results for the query '{query}'"] + [fact for _, _, fact in matches[:MAX_RESULTS]])


print(text_find.invoke({"query": "Halden Arc"}))
print()
print(text_find.invoke({"query": "a gadget for watching films on a wall"}))

In [ ]:
text_find_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, text_find],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
)


async def text_find_task(*, item, **kwargs):
    """Run the agent with the payout tool and the word lookup on one claim from the dataset."""
    return await decide(text_find_agent, item.input)

In [ ]:
text_find_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool and text-find",
        description="The agent with the payout tool and a word lookup in the knowledge base",
        data=items,
        task=text_find_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    text_find_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
text_find_table = pd.concat(text_find_tables, ignore_index=True)
show_confusion_matrix(text_find_table, "Payout tool and text-find")
text_find_table[~text_find_table["right"]]

In [ ]:
pd.DataFrame(
    [average_scores(payout_table), average_scores(memory_table), average_scores(text_find_table)],
    index=["Payout tool", "Payout tool and memory (qdrant-find)", "Payout tool and text-find"],
).round(3)

## Challenge 2: a memory about each customer

Each customer gets a Markdown file in `customer_notes`, named by their `customer_id`, such as `CUS-3310.md`. The agent has two tools: `read_customer_notes` reads a customer's file and `add_customer_note` adds a dated line to it. It reads the notes before it decides, and after it records its decision it adds a note for whoever handles the customer's next claim.

Each customer has only one claim in this set, so we start with three notes written by hand, from contact with the customers. Two of them matter to the claim and one doesn't:

- **CUS-3310** (CLM-1310, the break-in in Split): she has no receipts for the clothes. Section 5 asks for the police report and doesn't require receipts, so the claim can still be paid.
- **CUS-3325** (CLM-1325, the stitches in Oia): the itemized bill still hasn't arrived, and section 7 needs it.
- **CUS-3301** (CLM-1301): a contact preference, which shouldn't change anything.

**Between runs, the notes go back to these three.** With `N_TIMES` above 1, run 2 would otherwise read the notes that run 1 wrote about the same claims. It would be scored partly on its own earlier answers, and the runs would no longer be independent samples of the same agent. In production the notes build up from claim to claim; in an evaluation every run has to start from the same state.

In [ ]:
from datetime import date

CUSTOMER_NOTES = Path("customer_notes").resolve()

STARTING_NOTES = {
    "CUS-3301": "- 2026-05-14: Prefers to be contacted by email, not by phone.",
    "CUS-3310": "- 2026-08-12: Phone call. Hana Park says the swimsuit, the shirt and the sandals were bought for cash at a market, so she has no receipts for them.",
    "CUS-3325": "- 2026-08-18: Email. George Papadakis says the clinic in Oia still hasn't sent the itemized bill. He will forward it when it arrives.",
}


def reset_customer_notes():
    """Empty the customer notes folder and write the starting notes again."""
    shutil.rmtree(CUSTOMER_NOTES, ignore_errors=True)
    CUSTOMER_NOTES.mkdir()
    for customer_id, notes in STARTING_NOTES.items():
        (CUSTOMER_NOTES / f"{customer_id}.md").write_text(f"# Notes on {customer_id}\n\n{notes}\n", encoding="utf-8")


def notes_file(customer_id):
    """The notes file for a customer, or None if the id isn't a customer id like CUS-3310."""
    if not re.fullmatch(r"CUS-\d+", customer_id):
        return None
    return CUSTOMER_NOTES / f"{customer_id}.md"


@tool(parse_docstring=True)
def read_customer_notes(customer_id: str) -> str:
    """Read what UnderCover's handlers have noted about a customer: earlier contact, earlier claims and anything to watch for.

    Args:
        customer_id: The customer's id from the claim, such as CUS-3310
    """
    path = notes_file(customer_id)
    if path is None:
        return f"{customer_id} is not a customer id"
    if not path.exists():
        return f"There are no notes on {customer_id}"
    return path.read_text(encoding="utf-8")


@tool(parse_docstring=True)
def add_customer_note(customer_id: str, note: str) -> str:
    """Add a note about a customer for whoever handles their next claim.

    Args:
        customer_id: The customer's id from the claim, such as CUS-3310
        note: One or two sentences on what a colleague should know about this customer next time
    """
    path = notes_file(customer_id)
    if path is None:
        return f"{customer_id} is not a customer id"
    if not path.exists():
        path.write_text(f"# Notes on {customer_id}\n\n", encoding="utf-8")
    with path.open("a", encoding="utf-8") as notes:
        notes.write(f"- {date.today().isoformat()}: {note}\n")
    return f"Added a note on {customer_id}"


reset_customer_notes()
print(read_customer_notes.invoke({"customer_id": "CUS-3310"}))
print(read_customer_notes.invoke({"customer_id": "CUS-3302"}))

In [ ]:
CUSTOMER_LINE = "Before you decide, read the customer's notes. After you record your decision, add a note on anything a colleague should know about this customer on a future claim.\n"

customer_memory_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, text_find, read_customer_notes, add_customer_note],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}{CUSTOMER_LINE}\n{policy_wording}\n",
)


async def customer_memory_task(*, item, **kwargs):
    """Run the agent with the payout tool, the word lookup and the customer notes on one claim from the dataset."""
    return await decide(customer_memory_agent, item.input)

In [ ]:
customer_memory_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    reset_customer_notes()
    result = langfuse.run_experiment(
        name="Payout tool, text-find and customer notes",
        description="The agent with the payout tool, a word lookup in the knowledge base and a memory about each customer",
        data=items,
        task=customer_memory_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    customer_memory_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
customer_memory_table = pd.concat(customer_memory_tables, ignore_index=True)
show_confusion_matrix(customer_memory_table, "Payout tool, text-find and customer notes")
customer_memory_table[~customer_memory_table["right"]]

These are the notes after the last run: the three we started with, with whatever the agent added, and a few of the notes it wrote on customers who had none.

In [ ]:
for customer_id in ["CUS-3301", "CUS-3310", "CUS-3325", "CUS-3304", "CUS-3306"]:
    print(read_customer_notes.invoke({"customer_id": customer_id}))

In [ ]:
pd.DataFrame(
    [average_scores(payout_table), average_scores(memory_table), average_scores(text_find_table), average_scores(customer_memory_table)],
    index=["Payout tool", "Payout tool and memory (qdrant-find)", "Payout tool and text-find", "Text-find and customer notes"],
).round(3)

## Stretch challenge: a research sub-agent

The retrieval moves into a sub-agent, the research assistant. It has `qdrant-find` and the two customer notes tools, and the UnderCover agent calls it as a tool, `research_assistant`, with a request in plain English. The UnderCover agent no longer sees the memory tools at all: it asks a colleague and gets back what they found.

The UnderCover agent asks twice on each claim: once before it decides, for the customer's notes and the facts it needs, and once after it records its decision, to note anything for the customer's next claim.

Two things to know:

- **The sub-agent's context is small.** It doesn't get the policy wording, only its own instructions and the request, so each of its calls is cheap. The UnderCover agent keeps the wording, since it makes the decision.
- **`found_fact` needs the sub-agent's searches.** `decide` only sees the UnderCover agent's messages, and the research assistant replies in its own words. So the tool keeps whatever `qdrant-find` returned inside the sub-agent in `research_log`, and the task adds it to `retrieved` before scoring. In the same way it keeps the cost of the sub-agent's model calls in `research_costs`, so `llm_cost` covers both agents.

The tool takes LangChain's `config` and passes it on to the sub-agent, so in Langfuse the sub-agent's trace sits inside the UnderCover agent's tool call.

In [ ]:
from langchain_core.runnables import RunnableConfig

RESEARCH_INSTRUCTIONS = """You are the research assistant to UnderCover's claims handlers. You look things up for them; you don't decide claims.

You have two kinds of memory:
- qdrant-find searches UnderCover's knowledge base: news about storms and strikes, a catalogue of devices, and notes on dive sites, rivers and activity operators.
- read_customer_notes and add_customer_note read and add to the notes on a customer.

When you are asked to look something up, search for each product, event or place in the request, one at a time, and read the customer's notes if you are given a customer id. Reply with what you found, quoting the facts and the notes word for word. Say plainly when you found nothing.
When you are asked to note something about a customer, add it to their notes and confirm.
"""

research_log = []
research_costs = []

research_subagent = create_agent(
    model=luna,
    tools=[*memory_search, read_customer_notes, add_customer_note],
    system_prompt=RESEARCH_INSTRUCTIONS,
    middleware=[ToolRetryMiddleware(max_retries=3)],
)


@tool(parse_docstring=True)
async def research_assistant(request: str, config: RunnableConfig) -> str:
    """Ask the research assistant to look up products, events, places and a customer's notes, or to add a note about a customer.

    Args:
        request: What to look up, naming each product, event or place and the customer id, or the note to add and the customer id
    """
    result = await research_subagent.ainvoke({"messages": [{"role": "user", "content": request}]}, config=config)
    research_log.extend(message.text for message in result["messages"] if message.type == "tool" and message.name in SEARCH_TOOLS)
    research_costs.extend(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return result["messages"][-1].text


reset_customer_notes()
print(await research_assistant.ainvoke({"request": "What is the Halden Arc? Also read the notes on CUS-3310."}))

In [ ]:
RESEARCH_LINE = (
    "Before you decide, ask the research assistant about each product, event or place in the claim that you need to know more about, "
    "and for the customer's notes, giving the customer id. After you record your decision, ask it to note anything a colleague "
    "should know about this customer on a future claim.\n"
)

research_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, research_assistant],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{RESEARCH_LINE}\n{policy_wording}\n",
)


async def research_task(*, item, **kwargs):
    """Run the agent with the payout tool and the research sub-agent on one claim, adding the sub-agent's searches and its cost."""
    research_log.clear()
    research_costs.clear()
    answer = await decide(research_agent, item.input)
    retrieved = "\n".join([answer["retrieved"], *research_log])
    return {**answer, "retrieved": retrieved, "cost": answer["cost"] + sum(research_costs)}

In [ ]:
research_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    reset_customer_notes()
    result = langfuse.run_experiment(
        name="Research sub-agent",
        description="The agent with the payout tool and a research sub-agent that has qdrant-find and the customer notes",
        data=items,
        task=research_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    research_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
research_table = pd.concat(research_tables, ignore_index=True)
show_confusion_matrix(research_table, "Research sub-agent")
research_table[~research_table["right"]]

In Langfuse, open a trace from this run: the research assistant's own model calls and searches sit inside each `research_assistant` tool call.

In [ ]:
pd.DataFrame(
    [
        average_scores(payout_table),
        average_scores(memory_table),
        average_scores(text_find_table),
        average_scores(customer_memory_table),
        average_scores(research_table),
    ],
    index=["Payout tool", "Payout tool and memory (qdrant-find)", "Payout tool and text-find", "Text-find and customer notes", "Research sub-agent"],
).round(3)

## Conclusions

These come from one run of each experiment on the 25 claims, with Luna at `low`. Two runs of the same text-find agent scored 84% and 92%, so a difference of one or two claims between experiments is within the noise. The claims that fail in every run say more than the averages.

| Experiment | Accuracy | Recall | Kappa | Cost of mistakes | LLM cost |
|---|---|---|---|---|---|
| Payout tool | 0.52 | 0.647 | 0.178 | $2,903.60 | $0.011 |
| Payout tool and memory (qdrant-find) | 0.88 | 0.824 | 0.775 | $3,050.00 | $0.014 |
| Payout tool and text-find | 0.92 | 0.882 | 0.843 | $3,000.00 | $0.010 |
| Text-find and customer notes | 0.84 | 0.765 | 0.713 | $3,100.00 | $0.016 |
| Research sub-agent | 0.96 | 0.941 | 0.920 | $50.00 | $0.027 |

Precision was 1.0 in every run with a memory.

### A word lookup did as well as searching by meaning

`text-find` found the fact on all 19 claims that need one, as `qdrant-find` did, for less and with no server or embedding model. Every claim names the product, storm or place it turns on, and the agent searches by that name, so matching words is enough. Searching by meaning wins when the query and the fact share no words: for "a gadget for watching films on a wall", Qdrant put the Halden Arc first, and `text-find` returned only the Sentinel Wall dive site.

### Finding the fact was not the problem

With either search, CLM-1304 and CLM-1306 were declined wrongly in every run, with `found_fact` true. The agent wrote in its rationale that the strike was announced after the policy was bought, so the exclusion doesn't apply, and then recorded a decline anyway. Those two wrongful declines cost $3,000, nearly all of the cost of mistakes, and better retrieval couldn't fix them.

### Customer notes didn't help on this set, and they carry a risk

Each customer has only one claim here, so their notes can only matter on their next claim. Testing them properly needs a dataset with several claims per customer, in order. The note on CUS-3310, that there are no receipts for the clothes, didn't stop the agent referring CLM-1310: it gave the agent a fact, but its doubt was how to value the clothes.

The notes the agent wrote were mostly clear and useful, but they record its own conclusions as facts, so a wrong decision becomes a wrong note on the customer's next claim. On CUS-3304 it added a correction saying its own decline was inconsistent, since a decision can't be changed once it's recorded.

With `N_TIMES` above 1, the notes go back to the three starting notes before each run, so that no run reads what an earlier run wrote about the same claims.

### The research sub-agent fixed the wrongful declines

With the retrieval in a sub-agent, CLM-1304 and CLM-1306 were approved with the right payouts, and the cost of mistakes fell from about $3,000 to $50. The one mistake left is a needless referral of CLM-1310, now on whether a smart ring counts as electronics.

My reading is that the research is now a step of its own: the UnderCover agent gets the facts back quoted word for word and compares the dates before it records anything, rather than working it out inside the rationale. The traces in Langfuse are the place to check this. The sub-agent doubles the LLM cost, to about a tenth of a cent a claim, against $3,000 less in mistakes. It's one run, so the next step is `N_TIMES = 3` for the Qdrant agent and the sub-agent.

## Cleaning up Langfuse

This last step is optional. Set `CLEAN_UP = True` and run the cell to delete every trace in the Langfuse project your keys belong to, along with the experiments built on them, and the claims in the `even-harder-claims` dataset. This can't be undone. Only do it in a project you made for this course.

The empty `even-harder-claims` dataset stays, and running Part 4 again fills it. The run scores (accuracy, kappa and the rest) stay too. Langfuse's free plan allows 50 deletions of trace data a day, and each score would take one.

In [ ]:
def all_trace_ids():
    """Every trace id in the Langfuse project, collected from its observations a page at a time."""
    trace_ids = set()
    cursor = None
    while True:
        page = langfuse.api.observations.get_many(fields="core", limit=1000, cursor=cursor)
        trace_ids.update(observation.trace_id for observation in page.data)
        cursor = page.meta.cursor
        if not cursor:
            return list(trace_ids)


CLEAN_UP = False

if CLEAN_UP:
    trace_ids = all_trace_ids()
    for start in range(0, len(trace_ids), 1000):
        langfuse.api.trace.delete_multiple(trace_ids=trace_ids[start:start + 1000])

    dataset_items = langfuse.get_dataset(DATASET).items
    for item in dataset_items:
        langfuse.api.dataset_items.delete(id=item.id)

    print(f"Deleted {len(trace_ids)} traces and {len(dataset_items)} dataset items")